# LeetCode 1726 — Tuple with Same Product bằng Python

Link đề: https://leetcode.com/problems/tuple-with-same-product/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Counting

Skeleton LeetCode:

```python
class Solution:
    def tupleSameProduct(self, nums: List[int]) -> int:
        ...
```

Lộ trình: **4 vòng lặp** trên mọi bộ có thứ tự (`O(n⁴)`) → **3 vòng + tra tập hợp** (`O(n³)`) → **đếm tích của từng cặp bằng hash map** và công thức `8 · C(k, 2)` (`O(n²)`) → **sắp xếp danh sách tích** rồi đếm nhóm bằng nhau (`O(n² log n)`).

## 1. Mục tiêu học

1. Chuyển từ đếm **bộ có thứ tự** `(a, b, c, d)` sang đếm **cặp không thứ tự** `{a, b}` có cùng tích, và hiểu vì sao mỗi cặp-của-cặp sinh đúng **8** bộ.
2. Dùng **hash map đếm tần suất** (`Counter` / `defaultdict(int)`) trên `n(n−1)/2` cặp.
3. Áp dụng tổ hợp `C(k, 2) = k(k−1)/2` cho "chọn 2 cặp trong `k` cặp cùng tích".
4. Lập luận tại sao với các số **phân biệt**, hai cặp khác nhau có cùng tích thì **không chung phần tử** (nên điều kiện `a ≠ b ≠ c ≠ d` tự thỏa).
5. So sánh thực nghiệm `O(n⁴)`, `O(n³)`, `O(n²)` và kiểm chứng chéo trên input ngẫu nhiên.

## 2. Đề bài & ví dụ

Cho mảng `nums` gồm các số nguyên dương **đôi một phân biệt**. Trả về số **bộ** `(a, b, c, d)` sao cho `a · b = c · d`, trong đó `a, b, c, d` là các phần tử của `nums` và `a ≠ b ≠ c ≠ d` (bốn phần tử khác nhau).

**Ví dụ 1:** `nums = [2,3,4,6]` → `8`:
`(2,6,3,4), (2,6,4,3), (6,2,3,4), (6,2,4,3), (3,4,2,6), (4,3,2,6), (3,4,6,2), (4,3,6,2)`.

**Ví dụ 2:** `nums = [1,2,4,5,10]` → `16` (hai nhóm: `1·10 = 2·5` và `2·10 = 4·5`, mỗi nhóm 8 bộ).

**Ràng buộc:**
- `1 ≤ nums.length ≤ 1000`
- `1 ≤ nums[i] ≤ 10⁴`
- Các phần tử của `nums` đôi một phân biệt.

In [1]:
import random
import time
from collections import Counter, defaultdict
from itertools import combinations, permutations
from math import comb
from typing import List

examples = [
    ([2, 3, 4, 6], 8),
    ([1, 2, 4, 5, 10], 16),
]
for nums, want in examples:
    print(f"nums={nums} -> kỳ vọng {want}")

nums=[2, 3, 4, 6] -> kỳ vọng 8
nums=[1, 2, 4, 5, 10] -> kỳ vọng 16


## 3. Kiến thức nền

### 3.1 `itertools.combinations` và `permutations`

`combinations(nums, 2)` liệt kê cặp **không thứ tự** (mỗi cặp một lần); `permutations(nums, 4)` liệt kê bộ 4 **có thứ tự**, các phần tử khác vị trí.

In [2]:
print(list(combinations([2, 3, 4], 2)))
print(len(list(permutations([2, 3, 4, 6], 4))), "= 4!")

[(2, 3), (2, 4), (3, 4)]
24 = 4!


### 3.2 Đếm tần suất bằng `Counter`

In [3]:
prods = Counter(a * b for a, b in combinations([2, 3, 4, 6], 2))
print(prods)

Counter({12: 2, 6: 1, 8: 1, 18: 1, 24: 1})


### 3.3 Tổ hợp chập 2

Có `k` cặp cùng tích ⇒ số cách chọn **2 cặp khác nhau** là `C(k, 2) = k(k−1)/2`. Trong Python: `math.comb(k, 2)` hoặc `k * (k − 1) // 2`.

In [4]:
print([comb(k, 2) for k in range(1, 6)], [k * (k - 1) // 2 for k in range(1, 6)])

[0, 1, 3, 6, 10] [0, 1, 3, 6, 10]


### 3.4 Vì sao mỗi cặp-của-cặp cho 8 bộ

Với hai cặp `{a, b}` và `{c, d}` cùng tích: chọn cặp nào đứng trước (2 cách) × thứ tự trong cặp đầu (2) × thứ tự trong cặp sau (2) = **8**.

In [5]:
a, b, c, d = 2, 6, 3, 4
tuples = set()
for (x, y), (z, w) in [((a, b), (c, d)), ((c, d), (a, b))]:
    for p in [(x, y), (y, x)]:
        for q in [(z, w), (w, z)]:
            tuples.add(p + q)
print(len(tuples), sorted(tuples))

8 [(2, 6, 3, 4), (2, 6, 4, 3), (3, 4, 2, 6), (3, 4, 6, 2), (4, 3, 2, 6), (4, 3, 6, 2), (6, 2, 3, 4), (6, 2, 4, 3)]


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 2 `nums = [1,2,4,5,10]`: liệt kê tích của mọi cặp (10 cặp):

| cặp | tích | cặp | tích |
|---|---|---|---|
| 1·2 | 2 | 2·5 | **10** |
| 1·4 | 4 | 2·10 | **20** |
| 1·5 | 5 | 4·5 | **20** |
| 1·10 | **10** | 4·10 | 40 |
| 2·4 | 8 | 5·10 | 50 |

Tích `10` xuất hiện 2 lần, tích `20` xuất hiện 2 lần ⇒ mỗi tích cho `C(2,2) = 1` cách chọn hai cặp ⇒ `8 · (1 + 1) = 16`. ✓

**Câu hỏi:** liệu hai cặp cùng tích có thể chung phần tử, ví dụ `{a, b}` và `{a, d}`? Khi đó `a·b = a·d` ⇒ `b = d` (vì `a > 0`) ⇒ hai cặp trùng nhau. Vì các số phân biệt, hai **cặp khác nhau** cùng tích luôn gồm **4 số khác nhau**.

## 5. Cách 1 — Brute Force / baseline: duyệt mọi bộ có thứ tự

**Ý tưởng:** 4 vòng lặp chỉ số đôi một khác nhau, đếm khi `nums[i]·nums[j] == nums[k]·nums[l]`. Bám sát định nghĩa nhất.

In [6]:
def tuple_brute(nums):
    n = len(nums)
    cnt = 0
    for i in range(n):
        for j in range(n):
            if j == i: continue
            for k in range(n):
                if k == i or k == j: continue
                for l in range(n):
                    if l == i or l == j or l == k: continue
                    if nums[i] * nums[j] == nums[k] * nums[l]:
                        cnt += 1
    return cnt

for nums, want in examples:
    print(tuple_brute(nums), "kỳ vọng", want)

8 kỳ vọng 8
16 kỳ vọng 16


**Độ phức tạp:** Time `O(n⁴)`, Space `O(1)`. `n = 1000` ⇒ `10¹²` phép — bất khả thi.

**Nút thắt:** với `(a, b, c)` cố định, `d` được xác định duy nhất: `d = a·b / c`. Không cần vòng thứ tư.

In [7]:
random.seed(0)
for n in [10, 20, 30]:
    nums = random.sample(range(1, 200), n)
    t0 = time.perf_counter(); tuple_brute(nums)
    print(f"n={n}: {time.perf_counter() - t0:.3f}s")

n=10: 0.000s
n=20: 0.009s
n=30: 0.048s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1:** `d = a·b/c` ⇒ dùng **set** tra `d` trong `O(1)` ⇒ `O(n³)`.

**Insight 2 (quan trọng):** đẳng thức `a·b = c·d` chỉ liên quan tới **tích của cặp**. Gom mọi cặp không thứ tự theo tích:

`đáp án = Σ_{tích p} 8 · C(cnt[p], 2)`

với `cnt[p]` là số cặp `{x, y}` (x < y theo chỉ số) có `x·y = p`.

**Quy tắc cộng dồn tương đương (không cần lưu hết rồi mới tính):** khi gặp một cặp mới có tích `p`, nó tạo cặp-của-cặp với **mỗi** cặp cùng tích đã gặp trước ⇒ `ans += 8 · cnt[p]` rồi `cnt[p] += 1`. Tổng các `cnt` trước đó `0 + 1 + … + (k−1) = C(k, 2)`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Ba vòng + tra set (`O(n³)`)

Với mỗi bộ `(a, b, c)` khác nhau, `d = a·b/c` phải là **số nguyên**, có trong `nums`, và khác `a, b, c`.

In [8]:
def tuple_cubic(nums):
    s = set(nums)
    n = len(nums)
    cnt = 0
    for i in range(n):
        for j in range(n):
            if j == i: continue
            p = nums[i] * nums[j]
            for k in range(n):
                if k == i or k == j: continue
                c = nums[k]
                if p % c == 0:
                    d = p // c
                    if d in s and d != nums[i] and d != nums[j] and d != c:
                        cnt += 1
    return cnt

for nums, want in examples:
    print(tuple_cubic(nums), "kỳ vọng", want)

8 kỳ vọng 8
16 kỳ vọng 16


**Độ phức tạp:** Time `O(n³)`, Space `O(n)`. `n = 1000` ⇒ `10⁹` — vẫn quá chậm trong Python.

### 7.2 Cách 3 — Hash map đếm tích của từng cặp (`O(n²)`)

Từng đoạn:
1. `cnt = defaultdict(int)`.
2. Với mỗi cặp `i < j`: `p = nums[i] * nums[j]`; `ans += 8 * cnt[p]`; `cnt[p] += 1`.

In [9]:
def tuple_hash(nums):
    cnt = defaultdict(int)
    ans = 0
    n = len(nums)
    for i in range(n):
        a = nums[i]
        for j in range(i + 1, n):
            p = a * nums[j]
            ans += cnt[p]
            cnt[p] += 1
    return 8 * ans

for nums, want in examples:
    print(tuple_hash(nums), "kỳ vọng", want)

8 kỳ vọng 8
16 kỳ vọng 16


Mẹo nhỏ: cộng `cnt[p]` rồi nhân 8 **một lần** ở cuối thay vì nhân 8 trong vòng lặp.

**Độ phức tạp:** Time `O(n²)` trung bình (thao tác dict `O(1)`), Space `O(n²)` worst (mọi tích khác nhau — khoảng `5·10⁵` khóa khi `n = 1000`).

Phiên bản ngắn gọn dùng `Counter` + `combinations` (cùng độ phức tạp):

In [10]:
def tuple_counter(nums):
    c = Counter(a * b for a, b in combinations(nums, 2))
    return 8 * sum(k * (k - 1) // 2 for k in c.values())

for nums, want in examples:
    print(tuple_counter(nums), "kỳ vọng", want)

8 kỳ vọng 8
16 kỳ vọng 16


### 7.3 Cách 4 — Sắp xếp danh sách tích, đếm các nhóm bằng nhau

Không dùng hash: tạo list `n(n−1)/2` tích, sắp xếp, rồi quét các đoạn giá trị bằng nhau. Hữu ích khi hash bị tấn công (anti-hash) hoặc ngôn ngữ không có hash map tốt.

In [11]:
def tuple_sort(nums):
    prods = sorted(a * b for a, b in combinations(nums, 2))
    ans = 0
    i = 0
    m = len(prods)
    while i < m:
        j = i
        while j < m and prods[j] == prods[i]:
            j += 1
        k = j - i
        ans += k * (k - 1) // 2
        i = j
    return 8 * ans

for nums, want in examples:
    print(tuple_sort(nums), "kỳ vọng", want)

8 kỳ vọng 8
16 kỳ vọng 16


**Độ phức tạp:** Time `O(n² log n)` (sắp `~n²/2` phần tử), Space `O(n²)`.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 1 `[2,3,4,6]`:

| cặp | tích p | cnt[p] trước | ans += | cnt[p] sau |
|---|---|---|---|---|
| (2,3) | 6 | 0 | 0 | 1 |
| (2,4) | 8 | 0 | 0 | 1 |
| (2,6) | 12 | 0 | 0 | 1 |
| (3,4) | 12 | 1 | **1** | 2 |
| (3,6) | 18 | 0 | 0 | 1 |
| (4,6) | 24 | 0 | 0 | 1 |

`ans = 1` ⇒ đáp án `8 · 1 = 8`.

In [12]:
def trace_hash(nums):
    cnt = defaultdict(int); ans = 0
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            p = nums[i] * nums[j]
            before = cnt[p]
            ans += before; cnt[p] += 1
            mark = "  <- ghép với cặp cùng tích đã gặp" if before else ""
            print(f"({nums[i]:2d},{nums[j]:2d}) tích={p:3d} cnt trước={before} ans={ans}{mark}")
    print("đáp án = 8 *", ans, "=", 8 * ans)

trace_hash([2, 3, 4, 6])

( 2, 3) tích=  6 cnt trước=0 ans=0
( 2, 4) tích=  8 cnt trước=0 ans=0
( 2, 6) tích= 12 cnt trước=0 ans=0
( 3, 4) tích= 12 cnt trước=1 ans=1  <- ghép với cặp cùng tích đã gặp
( 3, 6) tích= 18 cnt trước=0 ans=1
( 4, 6) tích= 24 cnt trước=0 ans=1
đáp án = 8 * 1 = 8


In [13]:
def list_tuples(nums):
    """Liệt kê các bộ theo nhóm tích để đối chiếu với đề."""
    groups = defaultdict(list)
    for a, b in combinations(nums, 2):
        groups[a * b].append((a, b))
    out = []
    for p, pairs in groups.items():
        for (a, b), (c, d) in combinations(pairs, 2):
            for x, y in [(a, b), (b, a)]:
                for z, w in [(c, d), (d, c)]:
                    out.append((x, y, z, w)); out.append((z, w, x, y))
    return sorted(out)

res = list_tuples([1, 2, 4, 5, 10])
print(len(res)); print(res)

16
[(1, 10, 2, 5), (1, 10, 5, 2), (2, 5, 1, 10), (2, 5, 10, 1), (2, 10, 4, 5), (2, 10, 5, 4), (4, 5, 2, 10), (4, 5, 10, 2), (5, 2, 1, 10), (5, 2, 10, 1), (5, 4, 2, 10), (5, 4, 10, 2), (10, 1, 2, 5), (10, 1, 5, 2), (10, 2, 4, 5), (10, 2, 5, 4)]


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def tupleSameProduct(self, nums: List[int]) -> int:
        cnt = defaultdict(int)
        ans = 0
        n = len(nums)
        for i in range(n):
            a = nums[i]
            for j in range(i + 1, n):
                p = a * nums[j]
                ans += cnt[p]
                cnt[p] += 1
        return 8 * ans

sol = Solution()
for nums, want in examples:
    got = sol.tupleSameProduct(nums)
    print(got, "OK" if got == want else "SAI")

8 OK
16 OK


## 10. Vì sao thuật toán đúng

Gọi `P` là tập các cặp **không thứ tự** `{x, y}` gồm hai phần tử khác nhau của `nums`.

**Song ánh.** Mỗi bộ hợp lệ `(a, b, c, d)` xác định hai cặp `{a, b}`, `{c, d}` ∈ `P` khác nhau với cùng tích. Ngược lại, mỗi cặp-của-cặp `{{a, b}, {c, d}}` (hai cặp khác nhau cùng tích) sinh đúng 8 bộ có thứ tự (mục 3.4), và 8 bộ này đôi một khác nhau vì 4 số phân biệt.

**Bốn số phân biệt.** Nếu `{a, b} ≠ {c, d}` và `a·b = c·d`, giả sử chung phần tử `a = c` ⇒ `b = d` (chia cho `a > 0`) ⇒ hai cặp trùng — mâu thuẫn. Vậy `a, b, c, d` đôi một khác nhau, đúng điều kiện đề.

**Đếm.** Số cặp-của-cặp với tích `p` là `C(cnt[p], 2)` ⇒ đáp án `8 · Σ_p C(cnt[p], 2)`. Cách cộng dồn `ans += cnt[p]` trước khi tăng cho đúng `0 + 1 + … + (k−1) = C(k, 2)` với mỗi tích.

**Không tràn số:** `p ≤ 10⁸`; đáp án ≤ `8 · C(n²/2, 2)` — Python không lo tràn; C++/Java nên dùng `long` cho `ans` khi cần (thực tế đáp án nhỏ hơn nhiều).

## 11. Phân tích Time / Space

Gọi `n = len(nums) ≤ 1000`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Bốn vòng | `O(n⁴)` | `O(1)` | `n ≤ 30` |
| 2. Ba vòng + set | `O(n³)` | `O(n)` | `n ≤ 150` |
| 3. Hash map tích | `O(n²)` | `O(n²)` | **nộp bài** |
| 4. Sắp xếp tích | `O(n² log n)` | `O(n²)` | không cần hash |

In [15]:
random.seed(1)
nums = random.sample(range(1, 10**4 + 1), 1000)
for name, fn in [("hash", tuple_hash), ("counter", tuple_counter), ("sort", tuple_sort),
                 ("submit", Solution().tupleSameProduct)]:
    t0 = time.perf_counter(); r = fn(nums)
    print(f"{name:7s}: {r}  {time.perf_counter() - t0:.3f}s")

nums2 = list(range(1, 1001))     # nhiều tích trùng nhau
t0 = time.perf_counter(); r = tuple_hash(nums2)
print(f"nums = 1..1000: {r}  {time.perf_counter() - t0:.3f}s")

hash   : 87016  0.125s


counter: 87016  0.105s


sort   : 87016  0.181s


submit : 87016  0.109s
nums = 1..1000: 5894072  0.074s


## 12. Bộ test

Mảng ngẫu nhiên **phân biệt** (dùng `random.sample`), giá trị trong `[1, 10⁴]`; giá trị nhỏ để có nhiều tích trùng.

In [16]:
def tuple_perm(nums):
    """Kiểm chứng thứ hai: duyệt permutations(nums, 4) theo định nghĩa."""
    return sum(1 for a, b, c, d in permutations(nums, 4) if a * b == c * d)

fns = [tuple_cubic, tuple_hash, tuple_counter, tuple_sort, lambda a: Solution().tupleSameProduct(a)]

for nums, want in examples:
    assert tuple_brute(nums) == tuple_perm(nums) == want
    for fn in fns:
        assert fn(nums) == want

edge = [([1], 0), ([1, 2], 0), ([1, 2, 3], 0), ([1, 2, 3, 6], 8), ([1, 2, 3, 4, 6, 12], 40)]
for nums, want in edge:
    assert tuple_brute(nums) == want, nums
    for fn in fns:
        assert fn(nums) == want, (fn, nums)

random.seed(2024)
for _ in range(300):
    n = random.randint(1, 11)
    hi = random.choice([15, 30, 100, 10**4])
    nums = random.sample(range(1, hi + 1), min(n, hi))
    want = tuple_brute(nums)
    for fn in fns:
        assert fn(nums) == want, (fn, nums)
print("ví dụ + edge + 300 bộ ngẫu nhiên khớp brute force O(n^4)")

ví dụ + edge + 300 bộ ngẫu nhiên khớp brute force O(n^4)


In [17]:
random.seed(9)
for _ in range(30):
    nums = random.sample(range(1, 200), random.randint(1, 120))
    r = tuple_cubic(nums)
    assert r == tuple_hash(nums) == tuple_counter(nums) == tuple_sort(nums)
print("n tới 120: O(n^3) khớp các cách O(n^2)")

n tới 120: O(n^3) khớp các cách O(n^2)


Kiểm chứng edge `[1,2,3,4,6,12]`: các tích trùng là `6 = 1·6 = 2·3`, `12 = 1·12 = 2·6 = 3·4`, `24 = 2·12 = 4·6`. Số cặp-của-cặp: `C(2,2) + C(3,2) + C(2,2) = 1 + 3 + 1 = 5` ⇒ `40`. ✓ (đối chiếu bằng brute force ở trên).

## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên nhân 8 (hoặc nhân 4)

In [18]:
def wrong_no_8(nums):
    c = Counter(a * b for a, b in combinations(nums, 2))
    return sum(k * (k - 1) // 2 for k in c.values())

def wrong_times_4(nums):
    return 4 * wrong_no_8(nums)

for nums, want in examples:
    print(f"không nhân: {wrong_no_8(nums)}, nhân 4: {wrong_times_4(nums)} | đúng = {want}")

không nhân: 1, nhân 4: 4 | đúng = 8
không nhân: 2, nhân 4: 8 | đúng = 16


**Sửa:** mỗi cặp-của-cặp sinh **8** bộ có thứ tự (2 × 2 × 2).

### Lỗi 2 — Đếm cặp **có thứ tự** (i, j với i ≠ j) rồi vẫn nhân 8

Mỗi tích bị đếm gấp đôi số cặp ⇒ `C(2k, 2)` thay vì `C(k, 2)`.

In [19]:
def wrong_ordered_pairs(nums):
    c = Counter(nums[i] * nums[j] for i in range(len(nums)) for j in range(len(nums)) if i != j)
    return 8 * sum(k * (k - 1) // 2 for k in c.values())

for nums, want in examples:
    print("sai =", wrong_ordered_pairs(nums), "| đúng =", want)

sai = 80 | đúng = 8
sai = 144 | đúng = 16


**Sửa:** chỉ duyệt `i < j`. (Nếu đã đếm cặp có thứ tự thì công thức đúng là `Σ k(k−2)` với `k` là số cặp có thứ tự — dễ nhầm, nên tránh.)

### Lỗi 3 — Dùng `k²` hoặc `k` thay cho `C(k, 2)`

In [20]:
def wrong_k_square(nums):
    c = Counter(a * b for a, b in combinations(nums, 2))
    return 8 * sum(k * k for k in c.values() if k > 1)

nums = [1, 2, 3, 4, 6, 12]
print("sai =", wrong_k_square(nums), "| đúng =", tuple_hash(nums))

sai = 136 | đúng = 40


**Sửa:** số cách chọn **2 cặp khác nhau** trong `k` cặp là `k(k−1)/2`.

### Lỗi 4 — Ba vòng (Cách 2) dùng chia số thực `a*b/c`

`a*b/c` là `float`; với tích lớn có thể sai làm tròn, và `d in set` so sánh `float` với `int` dễ gây nhầm (ví dụ `7.0 in {7}` là `True` nhưng `6.999999` thì không). Ngoài ra quên kiểm tra **chia hết** ⇒ đếm `d` không nguyên.

In [21]:
def wrong_float(nums):
    s = set(nums); n = len(nums); cnt = 0
    for i in range(n):
        for j in range(n):
            if j == i: continue
            for k in range(n):
                if k in (i, j): continue
                d = nums[i] * nums[j] / nums[k]          # float
                if int(d) in s and int(d) not in (nums[i], nums[j], nums[k]):   # SAI: int() cắt phần lẻ
                    cnt += 1
    return cnt

nums = [2, 3, 5, 7]
print("sai =", wrong_float(nums), "| đúng =", tuple_brute(nums))

sai = 4 | đúng = 0


`2·5/3 = 3.33…` ⇒ `int` thành `3` có trong tập ⇒ đếm nhầm. **Sửa:** kiểm tra `p % c == 0` rồi dùng `p // c` (số nguyên), hoặc tốt hơn — dùng Cách 3.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đẳng thức dạng `f(a, b) = f(c, d)` ⇒ **gom theo giá trị `f` của cặp** bằng hash map — "meet in the middle" kiểu `O(n²)`.
- Đếm bộ có thứ tự ⇒ đếm cấu trúc không thứ tự rồi **nhân hệ số đối xứng** (ở đây 8).
- Cộng dồn `ans += cnt[key]; cnt[key] += 1` là cách đếm "số cặp bằng nhau" một lượt.
- Họ bài: 454 *4Sum II* (`a+b = −(c+d)`), 1 *Two Sum*, 1512 *Number of Good Pairs*, 2453…

## 15. Biến thể của bài

1. **Cho phép phần tử lặp** (không phân biệt): hai cặp cùng tích có thể chung giá trị nhưng khác chỉ số — phải cẩn thận hơn (đếm theo chỉ số, loại trường hợp chung chỉ số).
2. **Tổng thay vì tích:** `a + b = c + d` với số phân biệt — giống hệt, đổi `a*b` thành `a+b`.
3. **4Sum II (454):** bốn mảng, đếm `A[i]+B[j]+C[k]+D[l] = 0` ⇒ hash map tổng `A+B`.
4. **Đếm bộ không thứ tự** `{{a,b},{c,d}}`: bỏ hệ số 8.

In [22]:
def tuple_same_sum(nums):
    """Biến thể 2: a + b = c + d với số phân biệt (đối xứng tương tự)."""
    c = Counter(a + b for a, b in combinations(nums, 2))
    return 8 * sum(k * (k - 1) // 2 for k in c.values())

def tuple_same_sum_brute(nums):
    return sum(1 for a, b, c_, d in permutations(nums, 4) if a + b == c_ + d)

random.seed(3)
for _ in range(100):
    nums = random.sample(range(1, 30), random.randint(1, 8))
    assert tuple_same_sum(nums) == tuple_same_sum_brute(nums)
print("biến thể tổng: khớp brute force; [1,2,3,4] ->", tuple_same_sum([1, 2, 3, 4]))

def four_sum_ii(A, B, C, D):
    ab = Counter(a + b for a in A for b in B)
    return sum(ab[-(c + d)] for c in C for d in D)
print("4Sum II ví dụ:", four_sum_ii([1, 2], [-2, -1], [-1, 2], [0, 2]))

biến thể tổng: khớp brute force; [1,2,3,4] -> 8
4Sum II ví dụ: 2


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Bốn vòng | định nghĩa | `O(n⁴)` | `O(1)` | kiểm chứng |
| 2. Ba vòng + set | `d = ab/c` | `O(n³)` | `O(n)` | bước trung gian |
| 3. Hash map tích | `8·ΣC(k,2)` | `O(n²)` | `O(n²)` | **nộp bài** |
| 4. Sắp xếp tích | đếm nhóm bằng nhau | `O(n² log n)` | `O(n²)` | không muốn dùng hash |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `tupleSameProduct([1, 2, 3, 6])` và `tupleSameProduct([2, 3, 5])`?

**Bài 2 (tính tay):** với `[1,2,3,4,6,12]`, lập bảng các tích xuất hiện ≥ 2 lần và tính đáp án.

**Bài 3 (điền TODO):** hoàn thiện `count_unordered(nums)` trả về số cặp-của-cặp **không thứ tự** `{{a,b},{c,d}}` có cùng tích.

**Bài 4 (tự viết lại):** viết lại Cách 3 sao cho **chỉ** dùng `dict` thường (không `defaultdict`, không `Counter`).

**Bài 5 (mini challenge):** biến thể **cho phép phần tử lặp lại** trong `nums` (chỉ số khác nhau): đếm số bộ chỉ số `(i, j, k, l)` đôi một khác nhau với `nums[i]·nums[j] = nums[k]·nums[l]`. Viết lời giải `O(n²)` và kiểm chứng bằng brute force. (Biến thể tự đặt.)

In [23]:
def count_unordered(nums):
    # TODO: Counter tích các cặp, cộng C(k, 2)
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) `[1,2,3,6]`: `1·6 = 2·3`. (2) Một cặp-của-cặp ⇒ `8`. (3) `[2,3,5]` chỉ 3 phần tử ⇒ `0`.

**Bài 2:** (1) Tích 6, 12, 24. (2) Số cặp: 2, 3, 2. (3) `8·(1 + 3 + 1) = 40`.

**Bài 3:** (1) Giống Cách 3. (2) Không nhân 8. (3) Kiểm tra: `count_unordered(nums) * 8 == tupleSameProduct(nums)`.

**Bài 4:** (1) `cnt.get(p, 0)`. (2) `cnt[p] = cnt.get(p, 0) + 1`. (3) Nhân 8 ở cuối.

**Bài 5:** (1) Đếm cặp chỉ số `i < j` theo tích: `C(k, 2)` cặp-của-cặp — nhưng hai cặp có thể **chung chỉ số**. (2) Cặp `{i, j}` và `{i, l}` cùng tích ⇔ `nums[j] = nums[l]` (khi `nums[i] > 0`). (3) Trừ đi số cặp-của-cặp chung chỉ số: với mỗi `i` và mỗi giá trị `v` xuất hiện `m` lần ở các chỉ số khác `i`, có `C(m, 2)` cặp-của-cặp chung `i`.

## 19. Đáp án tham khảo

In [24]:
# Bài 1
assert Solution().tupleSameProduct([1, 2, 3, 6]) == 8 and Solution().tupleSameProduct([2, 3, 5]) == 0

# Bài 2
c = Counter(a * b for a, b in combinations([1, 2, 3, 4, 6, 12], 2))
assert {p: k for p, k in c.items() if k >= 2} == {6: 2, 12: 3, 24: 2}

# Bài 3
def count_unordered_ans(nums):
    c = Counter(a * b for a, b in combinations(nums, 2))
    return sum(k * (k - 1) // 2 for k in c.values())

# Bài 4
def tuple_plain_dict(nums):
    cnt = {}; ans = 0
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            p = nums[i] * nums[j]
            k = cnt.get(p, 0)
            ans += k; cnt[p] = k + 1
    return 8 * ans

random.seed(81)
for _ in range(200):
    nums = random.sample(range(1, 60), random.randint(1, 25))
    assert 8 * count_unordered_ans(nums) == tuple_hash(nums) == tuple_plain_dict(nums)
print("Bài 1-4: OK")

Bài 1-4: OK


In [25]:
# Bài 5 — cho phép giá trị lặp (đếm theo chỉ số)
def tuple_with_duplicates(nums):
    n = len(nums)
    c = Counter(nums[i] * nums[j] for i in range(n) for j in range(i + 1, n))
    total_pairs_of_pairs = sum(k * (k - 1) // 2 for k in c.values())
    # trừ các cặp-của-cặp chung một chỉ số i: {i, j} và {i, l} với nums[j] == nums[l], j != l, j, l != i
    shared = 0
    vc = Counter(nums)
    for i in range(n):
        for v, m in vc.items():
            m_other = m - (1 if nums[i] == v else 0)
            shared += m_other * (m_other - 1) // 2
    return 8 * (total_pairs_of_pairs - shared)

def brute_dup(nums):
    n = len(nums)
    return sum(1 for i, j, k, l in permutations(range(n), 4) if nums[i] * nums[j] == nums[k] * nums[l])

random.seed(82)
for _ in range(300):
    nums = [random.randint(1, 6) for _ in range(random.randint(1, 8))]
    assert tuple_with_duplicates(nums) == brute_dup(nums), nums
print("Bài 5: khớp brute force; [2,2,2,2] ->", tuple_with_duplicates([2, 2, 2, 2]), "(= 4! = 24)")

Bài 5: khớp brute force; [2,2,2,2] -> 24 (= 4! = 24)


Lưu ý Bài 5: hai cặp chung **cả hai** chỉ số là cùng một cặp, không được tính trong `C(k, 2)`; hai cặp chung **đúng một** chỉ số được trừ bằng `shared`. Vì giá trị dương, `nums[i]·nums[j] = nums[i]·nums[l]` ⇔ `nums[j] = nums[l]`, nên mọi cặp-của-cặp chung chỉ số đều có cùng tích — đếm `shared` như trên là chính xác.

## 20. Template nhớ nhanh + Checklist

```python
# Đếm bộ (a,b,c,d) với a*b == c*d, số phân biệt
cnt = defaultdict(int); ans = 0
for i in range(n):
    for j in range(i + 1, n):          # cặp không thứ tự
        p = nums[i] * nums[j]
        ans += cnt[p]                  # ghép với mọi cặp cùng tích trước đó
        cnt[p] += 1
return 8 * ans                         # 2 (thứ tự hai cặp) * 2 * 2
```

**Checklist trước khi submit:**
- [ ] Duyệt cặp `i < j` (không thứ tự).
- [ ] Công thức `C(k, 2)`, không phải `k²` hay `k`.
- [ ] Nhân **8** ở cuối.
- [ ] Không dùng chia số thực.
- [ ] `n < 4` ⇒ đáp án `0` (tự động đúng).